In [ ]:
import os
import copy
import time
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.datasets import ImageFolder
from torchvision.models import (DenseNet121_Weights,  ResNet50_Weights,  ResNet101_Weights, EfficientNet_B0_Weights)

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)
from preprocess_roi import preprocess_roi_from_uint16png

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class CTDataset(Dataset):
    def __init__(self, image_paths, labels, image_size=256):
        self.image_paths = image_paths
        self.labels = labels
        self.image_size = image_size
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        label = self.labels[idx]

        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        img = preprocess_roi_from_uint16png(img, size=self.image_size)

        img = torch.from_numpy(img).float().unsqueeze(0)
        img = img.repeat(3, 1, 1)

        img = (img - self.mean) / self.std

        label = torch.tensor(label, dtype=torch.long)
        return img, label

In [ ]:
train_dir = r"C:\Users\rgzep\Documents\IP data\stage2_nodule_crops\DataSplit\train"
val_dir   = r"C:\Users\rgzep\Documents\IP data\stage2_nodule_crops\DataSplit\val"
test_dir  = r"C:\Users\rgzep\Documents\IP data\stage2_nodule_crops\DataSplit\test"

img_size = 128
batch_size = 16
num_workers = 0

save_dir = "saved_models"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
train_base = ImageFolder(train_dir)
val_base = ImageFolder(val_dir)
test_base = ImageFolder(test_dir)

train_paths = [s[0] for s in train_base.samples]
train_labels = [s[1] for s in train_base.samples]

val_paths = [s[0] for s in val_base.samples]
val_labels = [s[1] for s in val_base.samples]

test_paths = [s[0] for s in test_base.samples]
test_labels = [s[1] for s in test_base.samples]

class_names = train_base.classes
num_classes = len(class_names)

print("Classes:", class_names)
print("Train:", len(train_paths))
print("Val:", len(val_paths))
print("Test:", len(test_paths))

In [ ]:
train_data = CTDataset(train_paths, train_labels, image_size=img_size)
val_data   = CTDataset(val_paths, val_labels, image_size=img_size)
test_data  = CTDataset(test_paths, test_labels, image_size=img_size)

train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True, num_workers=num_workers)
val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False, num_workers=num_workers)

In [ ]:
print(test_base.classes)

## Soft voting

In [ ]:
class WeightedSoftVotingEnsemble(nn.Module):
    def __init__(self, models, weights=None):
        super().__init__()

        self.models = nn.ModuleList(models)

        if weights is None:
            weights = [1.0] * len(models)

        weights = torch.tensor(weights, dtype=torch.float32)
        weights = weights / weights.sum()

        self.register_buffer("weights", weights)

        for model in self.models:
            model.eval()
            for p in model.parameters():
                p.requires_grad = False

    def forward(self, x):
        weighted_probs = torch.zeros(
            x.size(0),
            2,
            device=x.device
        )

        for model, weight in zip(self.models, self.weights):
            logits = model(x)
            probs = F.softmax(logits, dim=1)
            weighted_probs += weight * probs

        return weighted_probs

In [ ]:
resnet50 = models.resnet50(weights=None)
resnet50.fc = nn.Sequential(nn.Dropout(0.6), nn.Linear(resnet50.fc.in_features, num_classes))
resnet50.load_state_dict(torch.load("saved_models/resnet50_best_sen.pth", map_location=device))
resnet50.to(device)
resnet50.eval()

resnet101 = models.resnet101(weights=None)
resnet101.fc = nn.Sequential(nn.Dropout(0.6), nn.Linear(resnet101.fc.in_features, num_classes))
resnet101.load_state_dict(torch.load("saved_models/resnet101_best_sen.pth", map_location=device))
resnet101.to(device)
resnet101.eval()

densenet121 = models.densenet121(weights=None)
densenet121.classifier = nn.Sequential(nn.Dropout(0.6), nn.Linear(densenet121.classifier.in_features, num_classes))
densenet121.load_state_dict(torch.load("saved_models/densenet121_best_sen.pth",map_location=device))
densenet121.to(device)
densenet121.eval()

efficientnet = models.efficientnet_b0(weights=None)
in_features = efficientnet.classifier[1].in_features
efficientnet.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(in_features, num_classes))
efficientnet.load_state_dict(torch.load("saved_models/efficientnet_b0_best_sen.pth", map_location=device))
efficientnet.to(device)
efficientnet.eval()

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import torch
import torch.nn.functional as F

def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            probs = F.softmax(outputs, dim=1)[:,1]

            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    auc = roc_auc_score(all_labels, all_probs)
    recall = recall_score(all_labels, all_preds)

    return acc, f1, auc, recall

In [ ]:
def evaluate_ensemble(model, test_loader, device, class_names):
    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            probs = model(images)                 # weighted probabilities
            preds = torch.argmax(probs, dim=1)

            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

            # probability of positive class
            all_probs.extend(probs[:, 1].cpu().numpy())

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    # Metrics
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    auc = roc_auc_score(y_true, y_prob)

    cm = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names
    )

    print("Accuracy :", round(acc, 4))
    print("Precision:", round(prec, 4))
    print("Recall   :", round(rec, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC AUC  :", round(auc, 4))

    print("\nConfusion Matrix:")
    print(cm)

    print("\nClassification Report:")
    print(report)

    # ROC Curve
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)

    plt.figure(figsize=(7,6))
    plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
    plt.plot([0,1], [0,1], linestyle="--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.grid(True)
    plt.show()

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "auc": auc
    }

In [ ]:
def predict_ensemble(model, image_tensor, class_names):
    model.eval()
    image_tensor = image_tensor.to(device)

    with torch.no_grad():
        probs = model(image_tensor)
        confidence, predicted_class = torch.max(probs, dim=1)

    return {
        "prediction": class_names[predicted_class.item()],
        "confidence": confidence.item(),
        "probabilities": probs.cpu().numpy()
    }

In [ ]:
for name, model in [
    ("resnet50", resnet50),
    ("densenet121", densenet121),
    ("resnet101", resnet101),
    ("efficientnetb0", efficientnet)
]:
    acc, f1, auc, recall = evaluate(model, val_loader)
    print(name, acc, f1, auc, recall)

In [ ]:
#AUC weights
weights=[0.87,0.87,0.86]
ensemble_model = WeightedSoftVotingEnsemble(
    models=[resnet50, densenet121, resnet101],
    weights=weights
).to(device)
val_res = evaluate_ensemble(
    ensemble_model,
    val_loader,
    device,
    class_names
)
test_res = evaluate_ensemble(
    ensemble_model,
    test_loader,
    device,
    class_names
)

In [ ]:
#Acc weights
weights=[0.79, 0.78, 0.78]
ensemble_model = WeightedSoftVotingEnsemble(
    models=[resnet50, densenet121, resnet101],
    weights=weights
).to(device)
val_res = evaluate_ensemble(
    ensemble_model,
    val_loader,
    device,
    class_names
)
test_res = evaluate_ensemble(
    ensemble_model,
    test_loader,
    device,
    class_names
)

In [ ]:
#sensitivity
weights = [0.81,0.74,0.8]
ensemble_model = WeightedSoftVotingEnsemble(
    models=[resnet50, densenet121, resnet101],
    weights=weights
).to(device)
val_res = evaluate_ensemble(
    ensemble_model,
    val_loader,
    device,
    class_names
)
test_res = evaluate_ensemble(
    ensemble_model,
    test_loader,
    device,
    class_names
)

In [ ]:
#f1
weights = [0.83, 0.81, 0.83]
ensemble_model = WeightedSoftVotingEnsemble(
    models=[resnet50, densenet121, resnet101],
    weights=weights
).to(device)
val_res = evaluate_ensemble(
    ensemble_model,
    val_loader,
    device,
    class_names
)
test_res = evaluate_ensemble(
    ensemble_model,
    test_loader,
    device,
    class_names
)

In [ ]:
def get_single_model_probs(model, loader, device):
    model.eval()
    all_probs = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            logits = model(images)
            probs = torch.softmax(logits, dim=1)

            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.numpy())

    return np.vstack(all_probs), np.concatenate(all_labels)

In [ ]:
# 1. Collect each model's validation probabilities once
val_probs_list = []
models_list = [resnet50, densenet121, resnet101]
for model in models_list:
    probs, labels = get_single_model_probs(model, val_loader, device)
    val_probs_list.append(probs)

val_probs_array = np.stack(val_probs_list, axis=0)
# shape = [num_models, num_samples, num_classes]

In [ ]:
best = None

for w1 in np.arange(1, 10.5, 0.5):
    for w2 in np.arange(0, 2.1, 0.25):
        for w3 in np.arange(0, 2.1, 0.25):

            weights = np.array([w1, w2, w3])
            if weights.sum() == 0:
                continue

            weights = weights / weights.sum()

            ensemble_probs = np.tensordot(
                weights,
                val_probs_array,
                axes=(0, 0)
            )

            y_prob = ensemble_probs[:, 1]
            y_pred = np.argmax(ensemble_probs, axis=1)

            auc = roc_auc_score(labels, y_prob)
            f1 = f1_score(labels, y_pred, average="weighted")

            score = 0.9 * auc + 0.1 * f1

            if best is None or score > best["score"]:
                best = {
                    "weights": weights,
                    "auc": auc,
                    "f1": f1,
                    "score": score
                }

print(best)

In [ ]:
best_weights = best["weights"]

ensemble_model_fine_tune= WeightedSoftVotingEnsemble(
    models=models_list,
    weights=best_weights
).to(device)

evaluate_ensemble(ensemble_model_fine_tune, test_loader, device, class_names)

In [ ]:
from sklearn.metrics import recall_score

best_sen = None

for w1 in np.arange(1, 10.5, 0.5):
    for w2 in np.arange(0, 2.1, 0.25):
        for w3 in np.arange(0, 2.1, 0.25):

            weights = np.array([w1, w2, w3])
            if weights.sum() == 0:
                continue

            weights = weights / weights.sum()

            ensemble_probs = np.tensordot(
                weights,
                val_probs_array,
                axes=(0, 0)
            )

            y_prob = ensemble_probs[:, 1]
            y_pred = (y_prob >= 0.5).astype(int)  # explicit threshold

            auc = roc_auc_score(labels, y_prob)
            f1 = f1_score(labels, y_pred, average="weighted")

            # 🔥 malignant sensitivity (recall for class 1)
            sen = recall_score(labels, y_pred, pos_label=1)

            score = sen  # optimise purely for sensitivity

            if best_sen is None or score > best_sen["score"]:
                best_sen = {
                    "weights": weights,
                    "auc": auc,
                    "f1": f1,
                    "sensitivity": sen,
                    "score": score
                }

print(best_sen)

In [ ]:
best_sen_weights = best_sen["weights"]

ensemble_model_sen= WeightedSoftVotingEnsemble(
    models=models_list,
    weights=best_sen_weights
).to(device)


In [ ]:
evaluate_ensemble(ensemble_model_sen, val_loader, device, class_names)

In [ ]:
evaluate_ensemble(ensemble_model_sen, test_loader, device, class_names)

In [ ]:
#save best of exploratory or finetune options
torch.save(
    ensemble_model.state_dict(),
    "saved_models/ensemble_soft_voting_best.pth"
)

## Logistic Regression

In [ ]:
def collect_model_probs(models, loader):
    X_meta = []
    y_meta = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            batch_probs = []
            for model in models:
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                batch_probs.append(probs.cpu().numpy())

            batch_features = np.concatenate(batch_probs, axis=1)
            X_meta.append(batch_features)
            y_meta.append(labels.numpy())

    return np.vstack(X_meta), np.concatenate(y_meta)

In [ ]:
models = [resnet50, densenet121, resnet101]

for model in models:
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

In [ ]:
X_val, y_val = collect_model_probs(models, val_loader)
X_test, y_test = collect_model_probs(models, test_loader)

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_val, y_val)

In [ ]:
test_preds = meta_model.predict(X_test)
test_probs = meta_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, test_preds))
print("Precision:", precision_score(y_test, test_preds, average="macro", zero_division=0))
print("Recall   :", recall_score(y_test, test_preds, average="macro", zero_division=0))
print("F1 Score :", f1_score(y_test, test_preds, average="macro", zero_division=0))
print("AUC:", roc_auc_score(y_test, test_probs))
print(confusion_matrix(y_test, test_preds))